# 08 · Master Pipeline Kaggle

Pipeline completo: prepara scripts a partir do checkout do GitHub, detecta GPU, configura Google Drive para persistência/backup, instala ComfyUI com output local no SSD, sincroniza modelos selecionados manualmente, inicia ComfyUI no SSD local com health check e oferece push inicial condicional.


In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json

REPO_URL = "https://github.com/automadevs/colab-pipeline.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "colab-pipeline"
SCRIPTS_DIR = WORKDIR / "scripts"
COMFYUI_DIR = WORKDIR / "ComfyUI"
OUTPUT_DIR = COMFYUI_DIR / "output"
MODELS_DIR = COMFYUI_DIR / "models"
DATASET = "automamermaid/comfydocs"
DRIVE_BASE = "Automa/ComfyUI"

In [ ]:
# Sincroniza sempre o repositório como fonte da verdade e copia scripts para execução
import importlib
import shutil

if REPO_DIR.exists():
    print("[INFO] Atualizando repositório via git pull...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    print("[INFO] Clonando repositório...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# Copiar scripts atualizados do checkout garantindo que versões órfãs sejam eliminadas
if SCRIPTS_DIR.exists():
    shutil.rmtree(SCRIPTS_DIR)
shutil.copytree(REPO_DIR / "scripts", SCRIPTS_DIR)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

importlib.invalidate_caches()
for module_name in ("comfyui_setup", "ngrok_tunnel", "gpu_detect", "kaggle_drive_sync", "kaggle_sync"):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

print("[INFO] Scripts atualizados disponíveis:", sorted(p.name for p in SCRIPTS_DIR.glob("*.py")))

In [ ]:
# Detecção e validação de GPU NVIDIA, depois do checkout dos scripts
from gpu_detect import detect_gpu
GPU_INFO = detect_gpu()
print(json.dumps(GPU_INFO, indent=2, ensure_ascii=False))
if not GPU_INFO.get("has_gpu"):
    raise RuntimeError("GPU NVIDIA não detectada. Ative Accelerator → GPU no Kaggle antes de continuar.")

print("=== GPU / DISCO ===")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["df", "-h", str(WORKDIR)], check=False)

In [ ]:
# Configuração do Google Drive (rclone + service account) SOMENTE para persistência/backup
from kaggle_drive_sync import get_drive_path, setup_rclone_kaggle, test_drive_connection

print("=" * 60)
print("CONFIGURANDO GOOGLE DRIVE (PERSISTÊNCIA/BACKUP)")
print("=" * 60)

DRIVE_AVAILABLE = False
try:
    test_res = test_drive_connection(drive_base=DRIVE_BASE, env="kaggle")
    if test_res["status"] == "pass":
        DRIVE_AVAILABLE = True
        DRIVE_PATH = Path(test_res["drive_path"])
        print(f"[INFO] Google Drive pronto para sync em: {DRIVE_PATH}")
    else:
        print(f"[WARN] Google Drive não pôde ser montado: {test_res.get('error')}")
except Exception as e:
    print(f"[WARN] Falha ao configurar Google Drive: {e}")
    print("[INFO] O ComfyUI funcionará normalmente gerando no SSD local.")


In [ ]:
# Instala/atualiza ComfyUI + custom nodes usando output SEMPRE local no SSD
from comfyui_setup import setup_comfyui

CUSTOM_NODES = [
    "cubiq/ComfyUI_essentials",
    "lbouaraba/comfyui-krea2edit",
]

OUTPUT_DIR = COMFYUI_DIR / "output"

setup_comfyui(
    comfyui_dir=COMFYUI_DIR,
    models_dir=MODELS_DIR,
    custom_nodes=CUSTOM_NODES,
    output_dir=OUTPUT_DIR,
)
print(f"[INFO] ComfyUI configurado com output local no SSD: {OUTPUT_DIR}")


In [ ]:
from kaggle_sync import select_dataset_files, sync_dataset_to_local

AUTO_SELECT_SINGLE = True
CATEGORIES = ["checkpoints", "diffusion_models", "loras", "vae", "text_encoders", "clip", "controlnet", "upscale_models", "video_models", "embeddings"]

In [ ]:
# Seleção síncrona: Run All aguarda input() no próprio kernel.
SELECTED_FILES = select_dataset_files(
    dataset=DATASET,
    preselected_categories=CATEGORIES,
    auto_select_single=AUTO_SELECT_SINGLE,
)
print(f"[INFO] Seleção confirmada: {len(SELECTED_FILES)} arquivo(s)")

In [ ]:
# Download seletivo: somente os paths confirmados são enviados ao sync.
stats = sync_dataset_to_local(
    dataset=DATASET,
    target_dir=MODELS_DIR,
    selected_files=SELECTED_FILES,
    force=False,
)
print(json.dumps(stats, indent=2, ensure_ascii=False))

In [ ]:
# Inicia/reutiliza ComfyUI, valida a API e só então abre o ngrok
from comfyui_setup import start_comfyui_runtime

COMFYUI_PORT = 8188
runtime = start_comfyui_runtime(
    comfyui_dir=COMFYUI_DIR,
    host="127.0.0.1",
    port=COMFYUI_PORT,
    output_dir=OUTPUT_DIR,
    enable_ngrok=True,
)

if not runtime["health"]:
    raise RuntimeError(f"ComfyUI não ficou saudável. Log: {runtime['log_path']}")

print("=" * 60)
print("COMFYUI READY")
print(f"Local : {runtime['local_url']}")
print(f"Public: {runtime['public_url'] or '(ngrok indisponível)'}")
print(f"Output: {OUTPUT_DIR}")
print(f"GPU   : {GPU_INFO.get('gpu_count', 0)} GPU(s)")
print("=" * 60)
if runtime["proc"] is not None:
    print("PID:", runtime["proc"].pid)
else:
    print("PID: processo existente reutilizado")

In [ ]:
# Push inicial condicional: verifica se há outputs ou logs locais e oferece/executa sync
from kaggle_drive_sync import sync_outputs

existing_outputs = list(OUTPUT_DIR.glob("*")) if OUTPUT_DIR.exists() else []
existing_outputs = [f for f in existing_outputs if f.is_file()]
log_file = COMFYUI_DIR / "comfyui.log"
has_logs = log_file.exists() and log_file.stat().st_size > 0

if existing_outputs or has_logs:
    print(f"[INFO] Arquivos detectados: {len(existing_outputs)} output(s), log={has_logs}")
    cats = []
    if existing_outputs: cats.append("outputs")
    if has_logs: cats.append("logs")
    try:
        sync_res = sync_outputs(action="push", categories=cats, local_outputs=OUTPUT_DIR, drive_base=DRIVE_BASE, env="kaggle")
        print(f"[INFO] Push inicial concluído: {sync_res['synced']} enviado(s), {sync_res['skipped']} inalterado(s)")
    except Exception as e:
        print(f"[WARN] Push inicial não pôde ser executado: {e}")
else:
    print("[INFO] Nenhum arquivo em output/ ou log para envio no momento. Pronto para gerar!")


## Próximo passo

O ComfyUI está pronto e rodando localmente no SSD do Kaggle. Gere suas imagens normalmente pela interface ou API.

Para sincronizar os outputs gerados, workflows e logs com o Google Drive a qualquer momento, abra e execute o notebook manual:
**`09_sync_outputs.ipynb`**
